# Data Description
- Background: 民政局提供行政里 GIS 圖資，作為後續村里空間單位候選底圖；本次僅原始圖資與屬性說明。
- date received: 未知；本機下載／收到日期待學生或原始下載紀錄補證，不以文件建立日代填。
- Path to data file: [臺北市里界圖官方資料集](https://data.taipei/dataset/detail?id=6b17b31d-4e16-495e-95b1-9fd1f47c80d8)；來源網址由學生提供，2026-11-25 唯讀查閱詮釋資料，未重新下載原始資料。PDF 下載來源與本機位元組關係未認證。
- Unit of observation: 一列一個行政里面物件；本機 12 區、456 里，年度／版本未認證。
- Sample period: 本機適用年度未知。官方頁收錄範圍標示 2010-08-17 至 2025-06-17，不等同本機幾何有效期間。
- Known issues: 缺本機下載日期及歷史適用證據；AREA 定義未知；無效幾何數為 0 也不代表沒有重疊／縫隙或跨年邊界變動。
- Definition for each variable: 見 Summary Stats 的完整逐欄定義表；「觀測內容」與「官方定義」分列，官方未定義處明記未知。
- Data path: [G97_A_CAVLGE_P.shp](G97_A_CAVLGE_P.shp)、[G97_A_CAVLGE_P.shx](G97_A_CAVLGE_P.shx)、[G97_A_CAVLGE_P.dbf](G97_A_CAVLGE_P.dbf)、[G97_A_CAVLGE_P.prj](G97_A_CAVLGE_P.prj)、[G97_A_CAVLGE_P.cpg](G97_A_CAVLGE_P.cpg)；程式讀檔使用下方精確的工作包相對路徑。
- Role: `raw`（參考 PDF 亦逐檔登記）
- Verification status: 已實際執行唯讀文件化檢查；詳見保存輸出。檔案身分、基本結構及列明診斷可核對，不認證為 analysis-ready。
- SHA-256: 見下方固定綁定值及 Summary 逐檔大小／hash 表；每次讀檔前後重新計算。
- Next processing notebook: pending；尚未建立，不預設處理檔名。
- Author / contributors: 詹明翰、謝子詮、曾翊涵（學生確認三人共同完成本次 Unit 0.5；不推定個別分工）
- Created At: 2026-11-25（文件建立日，與 date received 不同）

**官方版本資訊與本機版本分開記錄：** 官方提供機關為民政局，資源更新 2026-09-21 14:26:43、詮釋資料更新 2026-09-21 14:27:00，說明採 TWD97。本機 PRJ 可辨識 EPSG:3826；未重新下載、比對官方當前包或認證 2024／2025 幾何相同。

本文件依 `templates/data/raw/template-doc-and-summary-stats.ipynb` 的原欄位與章節順序建立。
僅描述未修改原始資料；未刪除、補值、修復、正式轉換或輸出 processed。正式研究與規劃定義詳見 [PROJECT.md](../../PROJECT.md)。


In [1]:
from pathlib import Path
import hashlib, json, sys, platform, os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('Open within the project containing AGENTS.md and templates/.')

PROJECT_ROOT = find_project_root()
assert Path(sys.executable).resolve() == (PROJECT_ROOT / '.venv/Scripts/python.exe').resolve()
assert sys.prefix != sys.base_prefix
TEMP = PROJECT_ROOT / 'data/temp/unit05_documentation'
for variable, folder in {'MPLCONFIGDIR':'matplotlib', 'IPYTHONDIR':'ipython',
                         'JUPYTER_RUNTIME_DIR':'runtime'}.items():
    location = TEMP / folder
    location.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(location)
os.environ['PROJ_NETWORK'] = 'OFF'
pd.set_option('display.max_colwidth', 100)

def project_file(relative_path, area='raw'):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('Explicit project-relative path required')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('Input path outside permitted data area')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify_inputs(expected):
    rows = []
    for relative_path, digest in expected.items():
        path = project_file(relative_path)
        actual = sha256_file(path)
        if actual != digest:
            raise ValueError(f'Input changed; review provenance before loading: {relative_path}')
        rows.append({'Data path':relative_path, 'bytes':path.stat().st_size,
                     'format':path.suffix.lower(), 'SHA-256':actual})
    return pd.DataFrame(rows)

def show_table(title, frame, block=25):
    display(Markdown('**' + title + '**'))
    if not len(frame):
        display(Markdown('無符合紀錄。'))
    else:
        for start in range(0, len(frame), block):
            display(frame.iloc[start:start + block])

def sample_ten(title, frame, block_columns=12):
    display(Markdown('**' + title + '：實際前十列，按欄分組呈現；未抽樣或排序**'))
    for start in range(0, len(frame.columns), block_columns):
        display(frame.iloc[:10, start:start + block_columns])

def schema(frame, definitions, levels=None):
    rows=[]
    for column in frame.columns:
        meaning, evidence = definitions[column]
        rows.append({'欄位':column, '讀入型態':str(frame[column].dtype),
                     '定義或可觀察內容':meaning, '定義依據及限制':evidence,
                     '層級':levels.get(column, '原始紀錄') if levels else '原始紀錄',
                     '缺失數':int(frame[column].isna().sum()),
                     '非缺失相異值':int(frame[column].nunique(dropna=True))})
    return pd.DataFrame(rows)

def finish_check():
    for relative_path,digest in EXPECTED_SHA256.items():
        assert sha256_file(project_file(relative_path)) == digest, relative_path
    print({'input_hashes_unchanged':True, 'input_files_checked':len(EXPECTED_SHA256),
           'interpreter_matches_project_venv':True, 'python_version':platform.python_version(),
           'scope':'read-only raw documentation; no processed dataset or model output'})


## Change for Your Own Data
以下固定原始檔路徑與事前檢查所得 SHA-256；版本不同即停止，不換用其他檔案。
CSV 的編碼／代碼型態、Shapefile 的 reader 與配套檔均明列。讀入型態是本次 reader 的推斷或明示設定，不冒稱官方儲存型態。


In [2]:
EXPECTED_SHA256 = {'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824'}
INPUT_MANIFEST = verify_inputs(EXPECTED_SHA256)


## Summary


In [3]:
show_table('表 1：原始輸入逐檔檔案格式、大小及 SHA-256', INPUT_MANIFEST)
import geopandas as gpd
df=gpd.read_file(project_file('data/raw/G97_A_CAVLGE_P.shp'),engine='pyogrio',encoding='utf-8')
print({'rows':len(df),'attributes':len(df.columns)-1,'columns_including_geometry':len(df.columns),'encoding':'UTF-8 per CPG','CRS':df.crs.to_string()})


**表 1：原始輸入逐檔檔案格式、大小及 SHA-256**

,Data path,bytes,format,SHA-256
0,data/raw/G97_A_CAVLGE_P.shp,765784,.shp,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
1,data/raw/G97_A_CAVLGE_P.shx,3748,.shx,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
2,data/raw/G97_A_CAVLGE_P.dbf,272706,.dbf,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
3,data/raw/G97_A_CAVLGE_P.prj,399,.prj,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
4,data/raw/G97_A_CAVLGE_P.cpg,5,.cpg,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824


{'rows': 456, 'attributes': 28, 'columns_including_geometry': 29, 'encoding': 'UTF-8 per CPG', 'CRS': 'EPSG:3826'}


### Sample


In [4]:
sample_ten('表 3：里界原始屬性與幾何前十列',df)


**表 3：里界原始屬性與幾何前十列：實際前十列，按欄分組呈現；未抽樣或排序**

,AREA,NEW,FULL,PERF_ID,COUN_ID,CPID,CPTID,CPTVID,NPID,NPTID,NPTVID,PNAME
0,217699.73657,6.300500e+09,臺北市中正區龍福里,63005.0,6300500,63005,6300500,6300500020,63005.0,6300500.0,6.300500e+09,臺北市
1,224540.10144,6.300700e+09,臺北市萬華區福星里,63007.0,6300700,63007,6300700,6300700001,63007.0,6300700.0,6.300700e+09,臺北市
2,145214.88422,6.300300e+09,臺北市大安區龍坡里,63003.0,6300300,63003,6300300,6300300011,63003.0,6300300.0,6.300300e+09,臺北市
3,123617.91376,6.300700e+09,臺北市萬華區新起里,63007.0,6300700,63007,6300700,6300700004,63007.0,6300700.0,6.300700e+09,臺北市
4,605005.69464,6.300500e+09,臺北市中正區建國里,63005.0,6300500,63005,6300500,6300500022,63005.0,6300500.0,6.300500e+09,臺北市
5,200100.64787,6.300300e+09,臺北市大安區光武里,63003.0,6300300,63003,6300300,6300300010,63003.0,6300300.0,6.300300e+09,臺北市
6,139047.99298,6.300700e+09,臺北市萬華區西門里,63007.0,6300700,63007,6300700,6300700003,63007.0,6300700.0,6.300700e+09,臺北市
7,156615.37474,6.300500e+09,臺北市中正區新營里,63005.0,6300500,63005,6300500,6300500021,63005.0,6300500.0,6.300500e+09,臺北市
8,106572.13407,6.300300e+09,臺北市大安區古風里,63003.0,6300300,63003,6300300,6300300013,63003.0,6300300.0,6.300300e+09,臺北市
9,320963.70455,6.300700e+09,臺北市萬華區青山里,63007.0,6300700,63007,6300700,6300700006,63007.0,6300700.0,6.300700e+09,臺北市


,TNAME,VNAME,PTVNAME,PTNAME,TVNAME,TM2_MAX_X,TM2_MAX_Y,TM2_MIN_X,TM2_MIN_Y,MAX_X,MAX_Y,MIN_X
0,中正區,龍福里,臺北市中正區龍福里,臺北市中正區,中正區龍福里,302380.416,2769777.04,301710.965,2769173.26,121.51907,25.03520,121.51241
1,萬華區,福星里,臺北市萬華區福星里,臺北市萬華區,萬華區福星里,301503.796,2771369.61,300649.862,2770873.53,121.51044,25.04961,121.50196
2,大安區,龍坡里,臺北市大安區龍坡里,臺北市大安區,大安區龍坡里,303977.500,2768793.60,303604.212,2768291.93,121.53486,25.02627,121.53114
3,萬華區,新起里,臺北市萬華區新起里,臺北市萬華區,萬華區新起里,301268.389,2770557.48,300653.587,2770219.81,121.50808,25.04229,121.50197
4,中正區,建國里,臺北市中正區建國里,臺北市中正區,中正區建國里,302242.034,2770560.21,301112.162,2769749.62,121.51773,25.04228,121.50650
5,大安區,光武里,臺北市大安區光武里,臺北市大安區,大安區光武里,305384.073,2770891.27,304868.117,2770485.80,121.54888,25.04515,121.54375
6,萬華區,西門里,臺北市萬華區西門里,臺北市萬華區,萬華區西門里,301299.747,2770796.96,300682.073,2770463.99,121.50840,25.04445,121.50227
7,中正區,新營里,臺北市中正區新營里,臺北市中正區,中正區新營里,302730.127,2769749.62,302130.086,2768992.30,121.52253,25.03494,121.51656
8,大安區,古風里,臺北市大安區古風里,臺北市大安區,大安區古風里,303658.846,2768494.31,303257.683,2768061.35,121.53169,25.02358,121.52770
9,萬華區,青山里,臺北市萬華區青山里,臺北市萬華區,萬華區青山里,300439.999,2770576.98,299447.570,2769774.50,121.49987,25.04249,121.49001


,MIN_Y,SECT_NAME,LIE_NAME,sn,geometry
0,25.02977,中正區,龍福里,323,"POLYGON ((302000.322 2769174.593, 301977.218 2769235.287, 301977.026 2769235.79, 301964.014 2769..."
1,25.04516,萬華區,福星里,239,"POLYGON ((301021.54 2770959.999, 300949.739 2770978.444, 300886.977 2770994.567, 300807.047 2771..."
2,25.02175,大安區,龍坡里,381,"POLYGON ((303703.378 2768324.365, 303678.725 2768308.129, 303658.847 2768291.939, 303613.323 276..."
3,25.03926,萬華區,新起里,284,"POLYGON ((300959.923 2770257.309, 300946.131 2770259.683, 300877.178 2770271.553, 300874.835 277..."
4,25.03500,中正區,建國里,283,"POLYGON ((301127.679 2770041.996, 301141.266 2770090.308, 301153.166 2770132.619, 301177.691 277..."
5,25.04151,大安區,光武里,266,"POLYGON ((305104.795 2770491.751, 305017.806 2770493.606, 305016.533 2770493.633, 304958.927 277..."
6,25.04146,萬華區,西門里,273,"POLYGON ((300881.6 2770470.846, 300876.558 2770471.287, 300807.214 2770477.348, 300772.821 27704..."
7,25.02813,中正區,新營里,324,"POLYGON ((302511.257 2769183.329, 302475.373 2769236.498, 302466.026 2769250.145, 302449.458 276..."
8,25.01968,大安區,古風里,397,"POLYGON ((303532.733 2768172.586, 303463.915 2768107.456, 303417 2768063.056, 303415.305 2768061..."
9,25.03528,萬華區,青山里,282,"POLYGON ((300133.191 2769781.567, 299891.504 2769788.366, 299888.236 2769788.458, 299921.141 276..."


### Summary Stats


In [5]:
DEFINITIONS = {'AREA': ('原始面積數值欄；計算方法、單位、幾何版本及與目前面積一致性未驗證，不能直接作分析分母', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'NEW': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'FULL': ('本機觀測含市＋區＋里名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'PERF_ID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'COUN_ID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'CPID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'CPTID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'CPTVID': ('本機 10 碼且唯一的里識別字串；正式碼規格與人口 11 碼對應仍待確認', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'NPID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'NPTID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'NPTVID': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'PNAME': ('本機觀測為臺北市名稱字串', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'TNAME': ('本機觀測為行政區名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'VNAME': ('本機觀測為行政里名稱；含糖蔀里原字串', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'PTVNAME': ('本機觀測含市＋區＋里名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'PTNAME': ('本機觀測含市＋區名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'TVNAME': ('本機觀測含區＋里名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'TM2_MAX_X': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'TM2_MAX_Y': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'TM2_MIN_X': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'TM2_MIN_Y': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'MAX_X': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'MAX_Y': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'MIN_X': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'MIN_Y': ('官方頁列有此欄；正式定義、編碼或產製規則未知', '官方來源只有欄名清單；不從縮寫推定官方定義'), 'SECT_NAME': ('本機觀測為行政區名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'LIE_NAME': ('本機觀測為行政里名稱', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'sn': ('本機整數序號；排序與穩定 ID 意義未知', '本機觀測；官方欄位定義未取得，不等同正式碼表認證'), 'geometry': ('GeoPandas 由 SHP 讀入的幾何物件，非 DBF 原始屬性欄', 'SHP 實際讀取及 GeoPandas 資料結構')}
show_table('表 4：全部 28 個原始屬性欄＋geometry 的型態、定義及缺失',schema(df,DEFINITIONS),block=15)
show_table('表 5：原始屬性摘要；AREA 尚未驗證',df.drop(columns='geometry').describe(include='all').T)


**表 4：全部 28 個原始屬性欄＋geometry 的型態、定義及缺失**

,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
0,AREA,float64,原始面積數值欄；計算方法、單位、幾何版本及與目前面積一致性未驗證，不能直接作分析分母,本機觀測；官方欄位定義未取得，不等同正式碼表認證,原始紀錄,0,456
1,NEW,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,456
2,FULL,str,本機觀測含市＋區＋里名稱,本機觀測；官方欄位定義未取得，不等同正式碼表認證,原始紀錄,0,456
3,PERF_ID,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12
4,COUN_ID,str,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12
5,CPID,str,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12
6,CPTID,str,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12
7,CPTVID,str,本機 10 碼且唯一的里識別字串；正式碼規格與人口 11 碼對應仍待確認,本機觀測；官方欄位定義未取得，不等同正式碼表認證,原始紀錄,0,456
8,NPID,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12
9,NPTID,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,12


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
15,PTNAME,str,本機觀測含市＋區名稱,本機觀測；官方欄位定義未取得，不等同正式碼表認證,原始紀錄,0,12
16,TVNAME,str,本機觀測含區＋里名稱,本機觀測；官方欄位定義未取得，不等同正式碼表認證,原始紀錄,0,456
17,TM2_MAX_X,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,447
18,TM2_MAX_Y,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,445
19,TM2_MIN_X,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,448
20,TM2_MIN_Y,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,449
21,MAX_X,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,445
22,MAX_Y,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,447
23,MIN_X,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,441
24,MIN_Y,float64,官方頁列有此欄；正式定義、編碼或產製規則未知,官方來源只有欄名清單；不從縮寫推定官方定義,原始紀錄,0,449


**表 5：原始屬性摘要；AREA 尚未驗證**

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AREA,456.0,NaN,NaN,NaN,588925.3456,1351135.000678,31743.93631,134640.283548,210070.97093,428535.896548,16326540.39624
NEW,456.0,NaN,NaN,NaN,6300653091.179825,357880.57379,6300100002.0,6300300043.75,6300700003.5,6301000019.25,6301200042.0
FULL,456,456,臺北市中正區龍福里,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERF_ID,456.0,NaN,NaN,NaN,63006.530702,3.578805,63001.0,63003.0,63007.0,63010.0,63012.0
COUN_ID,456,12,6300300,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPID,456,12,63003,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPTID,456,12,6300300,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPTVID,456,456,6300500020,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NPID,456.0,NaN,NaN,NaN,63006.530702,3.578805,63001.0,63003.0,63007.0,63010.0,63012.0
NPTID,456.0,NaN,NaN,NaN,6300653.070175,357.880518,6300100.0,6300300.0,6300700.0,6301000.0,6301200.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SECT_NAME,456,12,大安區,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LIE_NAME,456,454,中央里,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sn,456.0,NaN,NaN,NaN,228.5,131.78012,1.0,114.75,228.5,342.25,456.0


#### Shapefile 組合、CRS 與幾何品質
SHP 儲存幾何；SHX 為幾何索引；DBF 儲存屬性；PRJ 為座標參考定義；CPG 指定屬性文字編碼。
五個配套檔均納入固定 hash，不只檢查 SHP。PRJ 為 TWD97／GRS80、Transverse Mercator，
中央經線 121°、假東距 250,000、比例因子 0.9999，平面單位公尺，可辨識 EPSG:3826。


In [6]:
import shapely
print({'CPG':project_file('data/raw/G97_A_CAVLGE_P.cpg').read_text(encoding='ascii'),
       'PRJ_WKT':project_file('data/raw/G97_A_CAVLGE_P.prj').read_text(encoding='ascii'),
       'reader_CRS':df.crs.to_string(),'EPSG':df.crs.to_epsg(),
       'coordinate_units':[axis.unit_name for axis in df.crs.axis_info],
       'geographic_datum':df.crs.datum.name,'projection_method':df.crs.coordinate_operation.method_name})
assert df.crs.to_epsg()==3826
show_table('表 6：12 行政區里數分布',df['TNAME'].value_counts().rename('里數').to_frame())
show_table('表 7：實際幾何類型',df.geom_type.value_counts().rename('筆數').to_frame())
show_table('表 8：原始欄位 AREA 摘要；未重算幾何面積',df[['AREA']].describe().T)
print({'village_rows':len(df),'districts':df['TNAME'].nunique(),'CPTVID_unique':bool(df['CPTVID'].is_unique),
       'CPTVID_missing':int(df['CPTVID'].isna().sum()),'CPTVID_lengths':df['CPTVID'].str.len().value_counts().to_dict(),
       'district_name_missing':int(df['TNAME'].isna().sum()),'village_name_missing':int(df['VNAME'].isna().sum()),
       'district_name_blank':int(df['TNAME'].fillna('').str.strip().eq('').sum()),
       'village_name_blank':int(df['VNAME'].fillna('').str.strip().eq('').sum()),
       'district_village_pair_duplicates':int(df.duplicated(['TNAME','VNAME']).sum()),
       'attribute_full_duplicates':int(df.drop(columns='geometry').duplicated().sum()),
       'null_geometry':int(df.geometry.isna().sum()),'empty_geometry':int(df.geometry.is_empty.sum()),
       'invalid_geometry':int((~df.geometry.is_valid).sum()),
       'exact_WKB_duplicate_rows':int(df.geometry.to_wkb().duplicated(keep=False).sum()),
       'native_coordinate_bounds':df.total_bounds.tolist()})
show_table('表 9：幾何有效性理由（未修復）',
           pd.Series(shapely.is_valid_reason(df.geometry.array)).value_counts().rename('筆數').to_frame())


{'CPG': 'UTF-8', 'PRJ_WKT': 'PROJCS["TWD_1997_TM_Taiwan",GEOGCS["GCS_TWD_1997",DATUM["D_TWD_1997",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",250000.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",121.0],PARAMETER["Scale_Factor",0.9999],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]', 'reader_CRS': 'EPSG:3826', 'EPSG': 3826, 'coordinate_units': ['metre', 'metre'], 'geographic_datum': 'Taiwan Datum 1997', 'projection_method': 'Transverse Mercator'}


**表 6：12 行政區里數分布**

,里數
TNAME,
大安區,53
士林區,51
文山區,43
北投區,42
中山區,42
信義區,41
內湖區,39
萬華區,36
松山區,33


**表 7：實際幾何類型**

,筆數
Polygon,456


**表 8：原始欄位 AREA 摘要；未重算幾何面積**

,count,mean,std,min,25%,50%,75%,max
AREA,456.0,588925.3456,1.351135e+06,31743.93631,134640.283548,210070.97093,428535.896548,1.632654e+07


{'village_rows': 456, 'districts': 12, 'CPTVID_unique': True, 'CPTVID_missing': 0, 'CPTVID_lengths': {10: 456}, 'district_name_missing': 0, 'village_name_missing': 0, 'district_name_blank': 0, 'village_name_blank': 0, 'district_village_pair_duplicates': 0, 'attribute_full_duplicates': 0, 'null_geometry': 0, 'empty_geometry': 0, 'invalid_geometry': 0, 'exact_WKB_duplicate_rows': 0, 'native_coordinate_bounds': [296266.05303084344, 2761514.89561711, 317197.2607379343, 2789176.1690160283]}


**表 9：幾何有效性理由（未修復）**

,筆數
Valid Geometry,456


#### 與 Unit 0.2 的關係及後續條件
456 里、12 區與唯一 CPTVID 均再度核對；本次新增確認 456 個 Polygon，無缺失、空或無效幾何。
這是幾何個體有效性診斷，不是全面拓樸驗證：重疊、縫隙、AREA 來源與歷史邊界仍待 Chapter 1。
事故須先確認其座標基準，再與本里界進行經審查的空間歸屬；人口先建立正式碼表證據及 crosswalk；
都市計畫須先補未知 CRS、版本與幾何／分類規則。本次沒有執行任何一項正式空間整合或重算里面積。


## Cross-Dataset Documentation — 僅規劃、未正式整合

| 原始資料 | 原始觀察單位／時間／空間粒度 | 預計連接與可支持變數 | 整合前條件 |
| --- | --- | --- | --- |
| 2024、2025 事故 | 2024 為當事人紀錄；2025 僅第一當事人；事故時分與點位 | 第一當事人候選事件；驗證 CRS 後以點在行政里面歸里；A 事故組成、C 負擔 | 官方事件口徑、照明欄位跨年映射、缺失／異常及座標基準；邊界點／未匹配規則另審查。 |
| 人口 | 年×月×區域×性別，含市／區／里 | 里層級「計」；12 月平均存量、人口結構 B1 | 代碼 crosswalk、糖廍里例外、月資料基準日與里界適用性；不視為交通暴露人口。 |
| 行政里界 | 456 個里面單元、單一未認證年代快照 | CPTVID 候選空間 ID；人口名稱及代碼驗證後連接，並作事故歸里 | 適用年度、幾何重疊／縫隙、原 AREA 定義與重算口徑；不因里數相同認證兩年幾何一致。 |
| 都市計畫 | 細計面與點、未認證年代快照、未知 CRS | 確認後以面圖層分類與疊合支援 B2 分區比例；點圖層用途待確認 | 原包、CRS、版本、143 筆無效幾何、重複及部分重疊、非一般分區、正式分類表。 |

| 規劃資料集 | 來源及變數 | 狀態與解釋限制 |
| --- | --- | --- |
| Dataset A | 事故；早／晚尖峰、深夜、週末、路口、追撞、側撞、人車、多人受傷組成 | 尚未建立；保留分子／有效分母／缺失／涵蓋率，不用總人口作事故組成分母。 |
| Dataset B | 人口 B1＋通過驗證的都市計畫 B2；其他道路環境 B3 尚未取得或選用 | 尚未建立；不納入事故結果，不以事故中的道路情境代表全里環境；分區不是實際土地使用。 |
| Dataset C | 事故數、傷亡、每件受傷數、事故空間密度、低樣本標記 | 尚未建立；24 小時與 2–30 日死亡分列，缺值不補成 0；每平方公里事故數不是暴露校正事故率。 |

village-year 為後續基礎資料結構；兩年比例合併須合計分子／有效分母再重算，
不能直接平均年度比例。零事故里及低樣本門檻仍待審查。本文件未建立 crosswalk、
年度人口表或 A／B／C，亦未執行事故歸里、土地使用疊合、分群或統計推論。

其他原始資料說明：
[事故](taipei_2024_2025_crashes_raw_documentation.ipynb)、[人口](taipei_population_age_raw_documentation.ipynb)、[里界](taipei_village_boundaries_raw_documentation.ipynb)、[都市計畫](taipei_urban_planning_raw_documentation.ipynb).


In [7]:
finish_check()


{'input_hashes_unchanged': True, 'input_files_checked': 5, 'interpreter_matches_project_venv': True, 'python_version': '3.12.2', 'scope': 'read-only raw documentation; no processed dataset or model output'}
